# فصل ۱۰: حافظه و نگهداری تاریخچه (Memory & History)
## بخش ۲: مدیریت خودکار تاریخچه با `RunnableWithMessageHistory`

در درسنامه قبل دیدیم که چطور با یک لیست ساده (`ChatMessageHistory`) پیام‌ها را ذخیره کنیم، اما آن روش دستی بود. اگر چند کاربر هم‌زمان با چت‌بات صحبت کنند:
1. **اتوماسیون:** نیاز به ابزاری داریم که ذخیره و بازیابی پیام‌ها را خودکار انجام دهد.
2. **تفکیک نشست‌ها (Multi-session):** باید تاریخچه هر کاربر بر اساس یک شناسه یکتا (`session_id`) از دیگران جدا نگهداری شود.

ابزار قدرتمند **`RunnableWithMessageHistory`** در LangChain دقیقاً برای حل همزمان این دو چالش طراحی شده است.

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# بارگذاری کلیدهای محیطی
load_dotenv()

# مقداردهی اولیه مدل
chat_model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0.7
)

print("✅ مدل و کتابخانه‌های مورد نیاز لود شدند!")

✅ مدل و کتابخانه‌های مورد نیاز لود شدند!


### ۱. ساخت قالب پرامپت با Placeholder برای تاریخچه

با استفاده از `("placeholder", "{chat_history}")` به LangChain اعلام می‌کنیم که تاریخچه پیام‌ها باید دقیقاً در این نقطه از مکالمه تزریق شود.

In [2]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "تو یک دستیار هوشمند و دوستانه هستی. کوتاه و دقیق پاسخ بده."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
])

chain = prompt | chat_model
print("✅ زنجیره اصلی ساخته شد!")

✅ زنجیره اصلی ساخته شد!


### ۲. تعریف منبع ذخیره‌سازی (`store`) و تابع بازیابی تاریخچه

یک دیکشنری به نام `store` ایجاد می‌کنیم و تابعی می‌نویسیم که بر اساس `session_id`، تاریخچه متناظر با آن را برگرداند (یا اگر وجود نداشت، یک نمونه جدید بسازد).

In [3]:
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """برگرداندن تاریخچه پیام‌ها بر اساس شناسه نشست (session_id)"""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

print("✅ تابع get_session_history آماده شد!")

✅ تابع get_session_history آماده شد!


### ۳. تبدیل زنجیره ساده به زنجیره با حافظه خودکار (`RunnableWithMessageHistory`)

- `chain`: زنجیره اصلی ما
- `get_session_history`: تابع ارائه‌دهنده تاریخچه
- `input_messages_key`: نام متغیر پیام کاربر در پرامپت (`"input"`)
- `history_messages_key`: نام متغیر تاریخچه در پرامپت (`"chat_history"`)

In [4]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

print("✅ زنجیره با حافظه خودکار آماده استفاده است!")

✅ زنجیره با حافظه خودکار آماده استفاده است!


d:\Project\LLM\llm-apps-course\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### ۴. شبیه‌سازی مکالمه با دو کاربر متفاوت («شیخ» و «مرید»)

ما با دو `session_id` مجزا پیام ارسال می‌کنیم تا ببینیم آیا حافظه هر کاربر تفکیک می‌شود یا خیر.

In [5]:
# تابع کمکی برای نمایش متن تمیز پاسخ
def get_text(resp):
    return resp.text if hasattr(resp, "text") and resp.text else resp.content

# مکالمه با کاربر اول (شیخ)
config_sheykh = {"configurable": {"session_id": "sheykh_123"}}
response1 = with_message_history.invoke(
    {"input": "سلام، اسم من شیخ است."},
    config=config_sheykh
)
print(f"AI to Sheykh: {get_text(response1)}")

# مکالمه با کاربر دوم (مرید)
config_morid = {"configurable": {"session_id": "morid_456"}}
response2 = with_message_history.invoke(
    {"input": "سلام، اسم من مرید است."},
    config=config_morid
)
print(f"AI to Morid: {get_text(response2)}")

C:\Users\aline\AppData\Local\Temp\ipykernel_22252\442248364.py:6: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = InMemoryChatMessageHistory()


AI to Sheykh: سلام شیخ عزیز! از آشنایی با شما خوشحالم. امروز چطور می‌توانم کمکتان کنم؟
AI to Morid: سلام مرید عزیز! خیلی خوشبختم. امروز چطور می‌تونم کمکت کنم؟


### ۵. آزمون حافظه: پرسیدن مجدد نام از کاربر اول (شیخ)

In [6]:
# پرسش مجدد با سشن شیخ
response3 = with_message_history.invoke(
    {"input": "اسم من چی بود؟"},
    config=config_sheykh
)
print(f"AI to Sheykh (Again): {get_text(response3)}")

AI to Sheykh (Again): اسم شما شیخ است.


### ۶. مشاهده تاریخچه ذخیره‌شده در `store` برای شیخ

می‌توانیم مستقیماً محتویات آبجکت `InMemoryChatMessageHistory` مربوط به کاربر را مشاهده کنیم:

In [7]:
sheykh_history = store["sheykh_123"]

print("--- تاریخچه مکالمه با شیخ ---")
for msg in sheykh_history.messages:
    print(f"{msg.type}: {msg.content}")

--- تاریخچه مکالمه با شیخ ---
human: سلام، اسم من شیخ است.
ai: [{'type': 'text', 'text': 'سلام شیخ عزیز! از آشنایی با شما خوشحالم. امروز چطور می\u200cتوانم کمکتان کنم؟', 'extras': {'signature': 'EmAKXgFpFH0T+hq4taozuSwDiIqvxRxSBYo5BryV2+NdhwAz5Dt4+Ljh18eWDzyCh3UNm26QH9CiQyBxtxwMFHV+CgUrVp5V32BvJwfs1qoNNnZnPUXnXI1ro6F/Rzoho88='}}]
human: اسم من چی بود؟
ai: [{'type': 'text', 'text': 'اسم شما شیخ است.', 'extras': {'signature': 'EmAKXgFpFH0T4vdqrRgfQUI05MEg0w7DrhBpY+XVyImqxYNUjXhucUCQqZeZWtFmpQZ52bXoxBe/Tm5Jlyuv7XXQ/DBMQY5gLUD2LhexIkQml1rR6XYEJ02KybAfGQ41oW4='}}]


### ۷. پیاده‌سازی سیستم چند چتی (مشابه ChatGPT)

یک کاربر ممکن است چندین موضوع گفتگوی جداگانه داشته باشد. با ترکیب شناسه کاربر و شناسه چت (`f"{user_id}_{chat_id}"`)، می‌توان چندین گفتگوی کاملاً مستقل برای یک کاربر ساخت:

In [8]:
# چت شماره ۱ کاربر علی: یادگیری پایتون
config_ali_chat1 = {"configurable": {"session_id": "ali_123_chat001"}}
resp_c1 = with_message_history.invoke(
    {"input": "من در این چت می‌خوام فقط در مورد زبان پایتون سوال بپرسم."},
    config=config_ali_chat1
)
print(f"Ali (Chat 1): {get_text(resp_c1)}")

# چت شماره ۲ کاربر علی: آشپزی
config_ali_chat2 = {"configurable": {"session_id": "ali_123_chat002"}}
resp_c2 = with_message_history.invoke(
    {"input": "سلام، دستور پخت قرمه‌سبزی رو بگو."},
    config=config_ali_chat2
)
print(f"Ali (Chat 2): {get_text(resp_c2)}")

# تست استقلال چت‌ها: سوال در چت ۱ درباره موضوع چت ۲
test_c1 = with_message_history.invoke(
    {"input": "موضوع اصلی چت ما چی بود؟"},
    config=config_ali_chat1
)
print(f"Ali (Chat 1 - Verification): {get_text(test_c1)}")

Ali (Chat 1): عالیه! آماده‌ام. هر سوالی در مورد پایتون داری بپرس.
Ali (Chat 2): سلام! قرمه‌سبزی خوشمزه این‌طوری درست میشه:

۱. **مواد لازم:** گوشت خورشتی، سبزی (تره، جعفری، گشنیز، کمی شنبلیله و اسفناج)، لوبیا قرمز (یا چیتی)، پیاز، لیمو عمانی، ادویه‌ها (زردچوبه، فلفل سیاه، نمک) و پودر لیمو.

۲. **سرخ کردن پیاز و گوشت:** پیاز رو نگینی خرد کرده و تفت بدید. گوشت رو اضافه کنید و با زردچوبه و فلفل تفت بدید تا رنگش عوض بشه.

۳. **پخت اولیه:** لوبیا (که از قبل خیس خورده) و چند لیوان آب جوش رو اضافه کنید و بذارید با حرارت ملایم بپزه.

۴. **سبزی:** سبزی‌ها رو ریز خرد کنید و خوب سرخ کنید (تا تیره بشه ولی نسوزه) و به گوشت نیم‌پز اضافه کنید.

۵. **جا افتادن:** لیمو عمانی‌ها رو سوراخ کنید و بندازید داخل خورش. نمک رو در انتهای پخت تنظیم کنید. 

خورش رو روی شعله ملایم بذارید تا خوب روغن بندازе و جا بیفته. نوش جان!
Ali (Chat 1 - Verification): زبان برنامه‌نویسی پایتون.


### 📌 جمع‌بندی مزایای `RunnableWithMessageHistory`

| قابلیت | روش دستی (`ChatMessageHistory`) | روش خودکار (`RunnableWithMessageHistory`) |
| :--- | :--- | :--- |
| **افزودن پیام ورودی/خروجی** | دستی با `add_user_message` و `add_ai_message` | کاملاً خودکار |
| **مدیریت چند کاربر (Multi-tenant)** | نیازمند کدهای شرطی و لیست‌های متعدد | پشتیبانی داخلی با `session_id` |
| **سازگاری با ساختار LCEL** | نیازمند هندل کردن دستی در هر فراخوانی | کامپوننت بومی و آماده اتصال به Chain |
| **توسعه‌پذیری به دیتابیس** | پیچیده و سخت | بسیار ساده (فقط با تغییر خروجی `get_session_history`) |